# Structural annotation 

## Python modules

In [ ]:
import scanpy as sc
import anndata as ad

import scanorama
import scvi

from scipy import sparse
import pandas as pd
import numpy as np

import pickle

import seaborn as sb
import matplotlib.pyplot as plt

import os

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

In [ ]:
%load_ext rpy2.ipython

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Warnings R
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR) 

## R modules

In [ ]:
%%R
library(Seurat)
library(dplyr)

## Plotting 

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [ ]:
# Matplot figures
plt.rcParams['figure.figsize']=(7, 7)

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [ ]:
# Set color function 
def set_color(adata, categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return(adata)

# Processing functions 

## Dim reduction and cluster function 

In [ ]:
def dim_clust(adata, use_rep=None, n_neighbors=15, resolution=1, min_dist=0.5, metric='euclidean', file=None): 
    
    # Dim reduction and clustering on the scanorama corrected embedding 
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    if resolution is not None: 
        sc.tl.leiden(adata, resolution=resolution)
    
    sc.tl.umap(adata, min_dist=min_dist)
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return(adata)

## Differential expression analysis 

In [ ]:
def DEA(adata, key): 
    
    # Normalize
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    
    # DEA Wilcoxon 
    sc.tl.rank_genes_groups(adata, key, method='wilcoxon', key_added="wilcoxon", use_raw=False)
    sc.pl.rank_genes_groups(adata, n_genes=25, fontsize=12, sharey=False, key="wilcoxon", ncols=5)

# Annotate

## Import data

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/structural/scvi_con.h5ad')

In [ ]:
adata_raw = adata.raw.to_adata()
adata = adata.raw.to_adata()
adata.raw = adata_raw

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
cell_type_genes_dict = {
    
    r'Immune cells': ['PTPRC'], 
    r'Fibroblast': ['THY1', 'MGP', 'MFAP5', 'COL18A1', 'APCDD1', 'LGR5', 'MYOC', 'PTGDS', 'COCH', 'MMP2', 'SPARC', 'TIMP2', 'CXCL2', 'CXCL3', 'IL6', 'APOE', 'CXCL12', 'CXCL14', 'JUN', 'ADH1B', 'RARRES2', 'COMP', 'TNC', 'COL11A1', 'POSTN', 'TNN', 'CRABP1'],
    r'Keratinocyte': ['KRT5', 'KRT14', 'KRT1', 'KRT10'], 
    r'Endothelial cell': ['PECAM1', 'CD74', 'LYVE1', 'CCL21'], 
    r'Pericyte': ['TAGLN', 'ACTA2', 'CCL19', 'CCL26'], 
    r'Melanocyte': ['PMEL', 'MLANA']
    
}

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=0.5, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/structural.h5ad')

In [ ]:
sc.tl.leiden(adata, resolution=0.1, restrict_to=('leiden', ['7']), key_added='leiden_restrict')
sc.tl.leiden(adata, resolution=0.1, restrict_to=('leiden_restrict', ['8']), key_added='leiden_restrict')
sc.tl.leiden(adata, resolution=0.1, restrict_to=('leiden_restrict', ['9']), key_added='leiden_restrict')
sc.tl.leiden(adata, resolution=0.1, restrict_to=('leiden_restrict', ['10']), key_added='leiden_restrict')

adata.obs['leiden'] = adata.obs['leiden_restrict']
del adata.obs['leiden_restrict']

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

## Fine Leiden annotation 

In [ ]:
leiden_cell_type_fine = {
    
    '0': r'$F_{ret, ECM}\ (0)$',
    '1': r'$F_{ret, immune}\ (1)$',
    '2': r'$F_{ret, activated}\ (2)$',
    '3': r'$F_{pap, ECM}\ (3)$',
    '4': r'$KC_{sb}\ (4)$',
    '5': r'$vEC\ (5)$',
    '6': r'$KC_{b}\ (6)$',
    '7,0': r'$Pc\ (7,0)$',
    '7,1': r'$Pc_{immune}\ (7,1)$',
    '8,0': r'$F_{DS}\ (8,0)$',
    '8,1': r'$F_{DP}\ (8,1)$',
    '9,0': r'$Mlc\ (9,0)$',
    '9,1': r'$Pc\ (9,1)$',
    '10,0': r'$lEC\ (10,0)$', 
    '10,1': r'$Mast\ cell\ (10,1)$' 

}

In [ ]:
# Mapper 
leiden_cell_type_fine = pd.DataFrame({'leiden_cell_type_fine': adata.obs['leiden'].map(lambda x: leiden_cell_type_fine.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_fine, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_fine'].cat.reorder_categories([
    
    r'$F_{ret, ECM}\ (0)$',
    r'$F_{ret, activated}\ (2)$',
    r'$F_{ret, immune}\ (1)$',
    r'$F_{pap, ECM}\ (3)$',
    r'$F_{DS}\ (8,0)$',
    r'$F_{DP}\ (8,1)$',
    r'$KC_{b}\ (6)$',
    r'$KC_{sb}\ (4)$',
    r'$vEC\ (5)$',
    r'$lEC\ (10,0)$', 
    r'$Pc\ (7,0)$',
    r'$Pc\ (9,1)$', 
    r'$Pc_{immune}\ (7,1)$',
    r'$Mlc\ (9,0)$', 
    r'$Mast\ cell\ (10,1)$'
    
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_fine')

## Main Leiden annotation 

In [ ]:
leiden_cell_type_main = {
    
    '0': r'$F_{ret, ECM}$',
    '1': r'$F_{ret, immune}$',
    '2': r'$F_{ret, activated}$',
    '3': r'$F_{pap, ECM}$',
    '4': r'$KC_{sb}$',
    '5': r'$vEC$',
    '6': r'$KC_{b}$',
    '7,0': r'$Pc$',
    '7,1': r'$Pc_{immune}$',
    '8,0': r'$F_{DS}$',
    '8,1': r'$F_{DP}$',
    '9,0': r'$Mlc$',
    '9,1': r'$Pc$',
    '10,0': r'$lEC$', 
    '10,1': r'$Mast\ cell$' 
  

}

In [ ]:
# Mapper 
leiden_cell_type_main = pd.DataFrame({'leiden_cell_type_main': adata.obs['leiden'].map(lambda x: leiden_cell_type_main.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$F_{ret, ECM}$',
    r'$F_{ret, activated}$',
    r'$F_{ret, immune}$',
    r'$F_{pap, ECM}$',
    r'$F_{DS}$',
    r'$F_{DP}$',
    r'$KC_{b}$',
    r'$KC_{sb}$',
    r'$vEC$',
    r'$lEC$', 
    r'$Pc$',
    r'$Pc_{immune}$',
    r'$Mlc$', 
    r'$Mast\ cell$'

    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_main')

## Remove Mast cells 

In [ ]:
adata = adata[adata.obs['leiden']!='10,1']

## Remove blood cells 

In [ ]:
adata = adata[adata.obs['hto_demux_class']=='Skin']

## Set colors 

In [ ]:
adata = set_color(adata, list(color.keys()))

In [ ]:
sc.pl.umap(adata, color=['leiden_cell_type_main'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

# Save result CSV

In [ ]:
adata.obs.loc[:, ['leiden_cell_type_main']].rename(columns={'leiden_cell_type_main':'cell_type_leiden_subset'}).to_csv('result/annotation/sbusets/structural_p.csv')

# Save AnnData object

In [ ]:
adata.write_h5ad('data/object/subsets/annotation/structural.h5ad')

# Save Seurat object

In [ ]:
adata_r = adata.copy()

In [ ]:
label_format = {
    
    r'$F_{ret, ECM}$': "F['ret, ECM']",
    r'$F_{ret, activated}$': "F['ret, activated']",
    r'$F_{ret, immune}$': "F['ret, immune']",
    r'$F_{pap, ECM}$': "F['pap, ECM']",
    r'$F_{DS}$': "F['DS']",
    r'$F_{DP}$': "F['DP']",
    r'$KC_{b}$': "KC['b']",
    r'$KC_{sb}$': "KC['sb']",
    r'$vEC$': "vEC",
    r'$lEC$': "lEC", 
    r'$Pc$': "Pc",
    r'$Pc_{immune}$': "Pc['immune']",
    r'$Mlc$': "Mlc"
    
}

adata_r.obs['leiden_cell_type_main'] = adata_r.obs['leiden_cell_type_main'].map(lambda x: label_format.get(x, x)).astype('category')

In [ ]:
label_format_order = [
    
    "F['ret, ECM']",
    "F['ret, activated']",
    "F['ret, immune']",
    "F['pap, ECM']",
    "F['DS']",
    "F['DP']",
    "KC['b']",
    "KC['sb']",
    "vEC",
    "lEC", 
    "Pc",
    "Pc['immune']",
    "Mlc"
    
]

In [ ]:
cnt = adata_r.X.T.todense()
cell_id = adata_r.obs_names
gene_id = adata_r.var_names
meta = adata_r.obs
umap = adata_r.obsm['X_umap']

In [ ]:
%%R -i cnt -i cell_id -i gene_id -i meta -i label_format_order -i umap

colnames(cnt) <- cell_id
rownames(cnt) <- gene_id

so <- Seurat::CreateSeuratObject(cnt, meta.data=meta, project="lymphocyte", assay="RNA")

colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- cell_id
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

so$leiden_cell_type_main <- factor(so$leiden_cell_type_main, levels=label_format_order)
so$leiden_cell_type_fine <- factor(paste(so$leiden_cell_type_main, paste0("*' (", so$leiden, ")'")))

write.csv(so@meta.data[, "leiden_cell_type_main", drop=FALSE] %>% dplyr::rename(cell_type_leiden_subset="leiden_cell_type_main"), "result/annotation/sbusets/structural_r.csv")

saveRDS(so, 'data/object/subsets/annotation/structural.rds')